# Week 04 — AI-assisted software development and code review

**Seismic Python & AI Engineering | Hands-on notebook**

Treat generative AI as a fast collaborator whose output must be specified, reviewed and tested. This module is offline: using any external model is optional.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Write prompts with explicit API and acceptance criteria.
- Use tests to audit generated code and prevent silent errors.
- Apply confidentiality and prompt-injection precautions.
- Build a deterministic input validator for model predictions.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. The AI-assisted development loop
Use this sequence: **specification → tests → generation → inspection → execution → review → commit**. Avoid the anti-pattern “ask a model for a big application, paste everything, and trust that it works.”

A useful engineering prompt states: domain context, inputs, outputs, units, error policy, API signature, dependencies, tests and constraints. Ask the model to state assumptions rather than silently deciding them.

## 2. A reusable prompt contract

> Act as a Python engineer reviewing a scientific package. Implement only `normalize_trace(trace)` according to these requirements: a finite nonempty 1D float array; return a new normalized array; keep zero traces zero; reject invalid inputs with `ValueError`; do not mutate the argument. Use NumPy only. Provide three pytest cases, including invalid input. Explain assumptions and do not modify unrelated files.

**Your turn A:** rewrite this prompt for `estimate_shift_samples(...)`, explicitly defining the **positive delay sign convention** and the upper lag bound.

## 3. Inspect deliberately flawed generated code
The snippet below is deliberately **wrong**, mimicking plausible AI output. Do not use it in your library. Identify at least five problems before reading the next section.

```python
def generated_normalize(trace):
    return trace / max(trace)
```

Issues to investigate include all-negative arrays, zeros, empty arrays, nonfinite values, NumPy's scalar semantics and mutation/copy policy. The short code is not necessarily the correct code.

In [ ]:
def flawed_normalize(trace):
    return trace / max(trace)

examples = [np.array([1., 2., -3.]), np.array([-1., -3.]), np.array([0., 0.])]
for example in examples:
    with np.errstate(divide="ignore", invalid="ignore"):
        print("input=", example, "output=", flawed_normalize(example))

## 4. Model output is untrusted input
Even if an LLM writes an inference pipeline, its outputs can contain wrong types, out-of-range classes, NaNs or unexpected shape. Write validation at the interface; model confidence is not an accuracy guarantee.

**Your turn B:** classify each candidate below as valid or invalid for a three-class classifier and explain why.

In [ ]:
candidates = {
    "valid": [0, 2, 1],
    "out_of_range": [0, 3, 1],
    "fractional": [0, 1.5, 1],
    "nan": [0, np.nan, 1],
    "wrong_shape": [[0], [1], [2]],
}
for name, candidate in candidates.items():
    print(name, np.asarray(candidate).shape)

## 5. Independent implementation — Week 4 challenge
Implement `qc_generated_predictions(predictions, n_samples, n_classes)`.

Return a 1D integer NumPy array of the requested length; labels must be finite integers in `[0, n_classes)`. Raise `ValueError` for incorrect shapes, fractional values, NaNs, out-of-range classes or invalid dimensions. Write tests **before** writing the body.

Never execute arbitrary code suggested by an unknown webpage, retrieved document or untrusted model output. Read it as text, inspect dependencies and run it in an isolated environment when permitted.

## 6. AI review log and data confidentiality
For an engineering task, record the date, your prompt or spec, generated proposal, what was accepted or rejected, the tests run and any unresolved risk. Avoid committing chat transcripts with restricted data.

**Confidentiality rule:** Do not upload laboratory code, proprietary seismic volumes, credentials, internal architecture or licensed information to public AI services without explicit authorization and an approved workflow. Prefer sanitized toy examples during study.

**Applied problem:** examine a hypothetical AI-generated patch that swaps baseline and monitor in the correlation call. Describe how a sign-convention regression test will catch the change even if plots still look plausible.

In [ ]:
from seismic_toolkit.attributes import estimate_time_shift
base = np.eye(1, 40, k=14).ravel()
mon = np.eye(1, 40, k=17).ravel()
correct = estimate_time_shift(base, mon, 0.001, 0.010)
reversed_inputs = estimate_time_shift(mon, base, 0.001, 0.010)
print("Correct sign:", correct, "Swapped inputs:", reversed_inputs)
assert np.isclose(correct, -reversed_inputs)

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week4 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week04.md`](../exercises/week04.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.